# 02 · Fundamentos de Spark: SparkSession, RDD y evaluación perezosa

**Objetivos de aprendizaje**

- Explicar la relación entre SparkSession, SparkContext, DataFrame y RDD.
- Observar cómo Spark divide los datos en particiones.
- Distinguir transformaciones y acciones, y comprobar la evaluación perezosa.
- Identificar transformaciones estrechas y amplias (*shuffle*) en el plan de ejecución.
- Aplicar operaciones de pares clave-valor y el conteo de palabras (WordCount).

| Aspecto | Detalle |
|---|---|
| Duración estimada | 35 minutos |
| Requisito previo | Cuaderno 01 |
| Entrada | Dataset activo (`data/dataset_config.json`) |
| Salida | Ninguna (cuaderno conceptual) |

### Cómo utilizar este cuaderno

1. Las celdas se ejecutan **en orden**, de arriba hacia abajo, con `Shift + Enter`. Cada celda puede depender de variables creadas en celdas anteriores.
2. Las celdas que comienzan con `# >>> CELDA EDITABLE` son las únicas que deben modificarse para adaptar el análisis a otro dataset. El resto del código se ajusta automáticamente a las columnas detectadas.
3. Cada sección sigue la misma estructura: **fundamento** (qué es y por qué se hace), **código**, **interpretación** del resultado y **adaptación** al dataset propio.
4. Ante un error `NameError` (variable no definida), lo habitual es que no se haya ejecutado una celda anterior. La solución es *Kernel > Restart Kernel and Run All Cells*.
5. Mientras la sesión de Spark está activa, la Spark UI (normalmente <http://localhost:4040>) muestra los *jobs*, *stages* y tareas ejecutados.

La guía completa de uso de los cuadernos se encuentra en `docs/08-uso-de-notebooks.md`.

## Preparación del entorno

La celda siguiente realiza cuatro tareas:

1. **Ubica la carpeta del taller** con `ubicar_taller()`, que busca `scripts/taller_spark.py` a partir de la carpeta actual. Esto evita errores de ruta del tipo `Input path does not exist: file:/data/...`, que aparecen cuando el cuaderno se abre desde una ubicación distinta, como ocurre en Google Colab, donde la carpeta de trabajo es `/content`. Después fija la carpeta `notebooks/` como carpeta de trabajo.
2. **Solo en Google Colab**: busca el taller en cualquier carpeta de `/content` y en Google Drive; si no lo encuentra, lo descarga con `git clone` desde la dirección indicada en `REPO_URL` (por eso, en Colab, debe reemplazarse `<usuario>` por el usuario de GitHub donde está publicado el taller). También instala las dependencias que falten en Colab.
3. Agrega la carpeta `scripts/` a la ruta de búsqueda de Python, para poder importar el módulo `taller_spark.py`, que contiene las funciones de apoyo del taller.
4. Importa `functions` de PySpark con el alias `F`, convención habitual en la documentación oficial (The Apache Software Foundation [ASF], 2026b).
Finalmente, crea la `SparkSession` mediante `iniciar_spark()`. Esta función verifica que Java sea la versión 17 o 21, utiliza el mismo intérprete de Python en el *driver* y en los *executors* y se conecta al clúster de Docker cuando existe la variable `SPARK_MASTER`; en cualquier otro caso trabaja en modo `local[*]`.

In [ ]:
# Preparación del entorno: se ejecuta siempre en primer lugar
import glob, os, subprocess, sys
from pathlib import Path

# >>> CELDA EDITABLE (solo Google Colab): dirección del repositorio del taller en GitHub
REPO_URL = "https://github.com/<usuario>/taller-apache-spark.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def ubicar_taller():
    """Devuelve la carpeta raíz del taller: la que contiene scripts/taller_spark.py."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for patron in ("/content/*/scripts/taller_spark.py",                  # clonado en Colab
                   "/content/drive/MyDrive/*/scripts/taller_spark.py",    # copiado en Google Drive
                   "/content/drive/MyDrive/*/*/scripts/taller_spark.py"):
        candidatos += [Path(p).parent.parent for p in sorted(glob.glob(patron))]
    for base in candidatos:
        if (base / "scripts" / "taller_spark.py").exists():
            return base
    if en_colab():
        if "<usuario>" in REPO_URL:
            raise RuntimeError("En Google Colab debe escribirse en REPO_URL la dirección del repositorio "
                               "del taller (reemplazar <usuario>) y volver a ejecutar esta celda.")
        destino = Path("/content") / Path(REPO_URL.rstrip("/")).stem        # nombre del repositorio
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encontró la carpeta del taller (scripts/taller_spark.py). "
                            "Abrir el cuaderno desde la carpeta notebooks/ del repositorio.")

RAIZ_TALLER = ubicar_taller()
os.chdir(RAIZ_TALLER / "notebooks")              # rutas iguales en cualquier entorno (local, Docker, Colab)
sys.path.insert(0, str(RAIZ_TALLER / "scripts"))

if en_colab():                                    # en Colab se instalan las dependencias que falten
    import importlib.util
    faltan = [m for m in ("openpyxl",) if importlib.util.find_spec(m) is None]
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) < 4:
            faltan.append("pyspark==4.3.0")
    except ImportError:
        faltan.append("pyspark==4.3.0")
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Carpeta del taller:", RAIZ_TALLER)

from taller_spark import *                       # iniciar_spark, cargar_desde_config, perfil, ...
from pyspark.sql import functions as F           # funciones de columnas de Spark

spark = iniciar_spark("02_Fundamentos")

## Carga del dataset activo

El dataset activo es el que se configuró en el cuaderno `01_configuracion_dataset.ipynb` y quedó guardado en `data/dataset_config.json`. Si no se ha configurado ninguno, se utiliza el dataset de ejemplo (`data/ejemplo/ventas_tienda.csv`).

La función `clasificar_columnas()` asigna a cada columna un rol, que los cuadernos usan para decidir qué análisis aplicar:

| Variable | Rol | Criterio |
|---|---|---|
| `NUM` | numéricas | tipos numéricos que no son identificadores |
| `CAT` | categóricas | texto o booleanos con pocos valores distintos (hasta `max_categorias`) |
| `FEC` | fechas | tipos `date` o `timestamp` |
| `TXT` | texto | texto con muchos valores distintos (comentarios, descripciones) |
| — | identificadores | columnas excluidas en la configuración o con valores casi únicos |

Con un dataset propio, conviene revisar esta clasificación. Si una columna quedó en un rol inadecuado, puede corregirse en la configuración (`columnas_excluir`, `max_categorias`) o directamente en las listas, por ejemplo `CAT.remove("columna")`.

In [ ]:
# Carga del dataset activo (definido y guardado en el cuaderno 01)
df, cfg = cargar_desde_config(spark, version="original")

# Clasificación automática de columnas por rol analítico
roles = clasificar_columnas(df, excluir=cfg["columnas_excluir"], max_categorias=cfg["max_categorias"])
NUM, CAT, FEC, TXT = roles["numericas"], roles["categoricas"], roles["fechas"], roles["texto"]
objetivo = columna_objetivo(cfg, df)
resumen_roles(roles)
print("Columna objetivo:", objetivo)

## 1. De la SparkSession al RDD

La `SparkSession` es el punto de entrada a Spark. Contiene un `SparkContext` (`spark.sparkContext`), que coordina la ejecución en el clúster. Los datos pueden manejarse con dos abstracciones:

- **RDD** (*Resilient Distributed Dataset*): colección inmutable de objetos distribuida en particiones. Es la abstracción original de Spark (Zaharia et al., 2012).
- **DataFrame**: colección distribuida con columnas con nombre y tipo, que Spark optimiza mediante el optimizador Catalyst (Armbrust et al., 2015).

Todo DataFrame se ejecuta internamente sobre RDD. El atributo `df.rdd` expone esa representación: cada elemento es un objeto `Row`.

In [ ]:
sc = spark.sparkContext
rdd = df.rdd
print("Tipo de objeto     :", type(rdd).__name__)
print("Primer elemento    :", rdd.first())

## 2. Particiones

Una **partición** es un bloque de datos que se procesa de forma independiente. Spark ejecuta una tarea (*task*) por partición y reparte las tareas entre los núcleos disponibles; por eso el número de particiones determina el grado de paralelismo (ASF, 2026a).

In [ ]:
print("Núcleos disponibles (defaultParallelism):", sc.defaultParallelism)
print("Particiones del dataset:", rdd.getNumPartitions())

# Cantidad de registros en cada partición
por_particion = rdd.glom().map(len).collect()
print("Registros por partición:", por_particion)

**Interpretación.** Un archivo pequeño suele quedar en una o pocas particiones; los archivos grandes se dividen en bloques de aproximadamente 128 MB. `repartition(n)` redistribuye los datos en `n` particiones (con *shuffle*), mientras que `coalesce(n)` reduce el número de particiones sin redistribuir por completo.

In [ ]:
df4 = df.repartition(4)
print("Después de repartition(4):", df4.rdd.glom().map(len).collect())

## 3. Transformaciones y acciones

Las **transformaciones** (`map`, `filter`, `select`, `groupBy`...) describen un nuevo conjunto de datos a partir de otro, pero **no se ejecutan** de inmediato: Spark solo registra el plan. Las **acciones** (`count`, `collect`, `show`, `take`, `write`...) piden un resultado y desencadenan la ejecución de todo el plan. Este mecanismo, llamado **evaluación perezosa**, permite optimizar el plan completo antes de ejecutarlo (Zaharia et al., 2012).

La celda siguiente define una transformación que tarda deliberadamente 1 milisegundo por registro sobre una muestra de 2 000 registros.

In [ ]:
import time

def lenta(fila):
    time.sleep(0.001)
    return fila

muestra = df.limit(2000).rdd

t = time.time()
transformado = muestra.map(lenta)                 # transformación
print(f"Definir la transformación: {time.time() - t:.3f} s")

t = time.time()
n = transformado.count()                          # acción
print(f"Ejecutar la acción count(): {time.time() - t:.3f} s ({n} registros)")

**Interpretación.** Definir la transformación es prácticamente instantáneo porque no se procesa ningún dato. El tiempo real aparece al llamar a la acción `count()`.

### Linaje

Spark registra la secuencia de transformaciones que produjo cada RDD. Si se pierde una partición por un fallo, la reconstruye repitiendo esos pasos; de ahí el término *resilient* (Zaharia et al., 2012).

In [ ]:
print(transformado.toDebugString().decode())

## 4. Pares clave-valor con RDD

Muchas operaciones de Big Data consisten en agrupar por una clave y combinar valores. Con RDD, el patrón clásico es `map` (crear pares `(clave, 1)`) seguido de `reduceByKey` (sumar por clave).

El código siguiente cuenta los registros por cada valor de la primera columna categórica del dataset activo.

**Adaptación.** La columna se elige automáticamente (`CAT[0]`). Para usar otra, se cambia el valor de `COLUMNA` en la celda editable.

In [ ]:
# >>> CELDA EDITABLE: columna categórica para el conteo
COLUMNA = CAT[0] if CAT else None
print("Columna seleccionada:", COLUMNA)

In [ ]:
if COLUMNA:
    conteo_rdd = (df.select(COLUMNA).rdd
                  .map(lambda fila: (fila[0], 1))          # (valor, 1)
                  .reduceByKey(lambda a, b: a + b)          # suma por valor (provoca shuffle)
                  .sortBy(lambda par: par[1], ascending=False))
    for valor, n in conteo_rdd.take(10):
        print(f"{str(valor):<25}{n:>8,}")
else:
    print("El dataset no tiene columnas categóricas.")

El mismo resultado se obtiene con la API de DataFrames, en una sola línea y con mejor rendimiento, porque Catalyst optimiza la consulta y el motor evita convertir cada fila en un objeto de Python (Armbrust et al., 2015).

In [ ]:
if COLUMNA:
    df.groupBy(COLUMNA).count().orderBy(F.desc("count")).show(10, truncate=False)

`reduceByKey` combina los valores dentro de cada partición **antes** del *shuffle*, por lo que transfiere menos datos por la red que `groupByKey`. Por esta razón se prefiere siempre que sea posible (Chambers & Zaharia, 2018).

## 5. Transformaciones estrechas y amplias

- **Estrechas** (`filter`, `select`, `withColumn`): cada partición de salida depende de una sola partición de entrada; no hay movimiento de datos.
- **Amplias** (`groupBy`, `join`, `orderBy`, `distinct`): requieren un **shuffle**, es decir, redistribuir datos entre particiones. Son más costosas y separan el trabajo en *stages*.

El método `explain()` muestra el plan físico. En el plan siguiente, la operación `Exchange` corresponde al *shuffle* producido por `groupBy`.

In [ ]:
if COLUMNA:
    df.filter(F.col(COLUMNA).isNotNull()).groupBy(COLUMNA).count().explain()

**Actividad en la Spark UI.** Al abrir la pestaña *Jobs* y seleccionar el último trabajo, el grafo (DAG) muestra dos *stages* separadas por el *shuffle*.

## 6. Persistencia en memoria (cache)

Cuando un DataFrame se reutiliza varias veces, Spark lo recalcula desde el origen en cada acción. `cache()` lo conserva en memoria tras la primera ejecución, lo que acelera las siguientes.

In [ ]:
def medir(accion):
    t = time.time(); accion(); return time.time() - t

print(f"Sin cache            : {medir(lambda: df.count()):.3f} s")
df.cache()
print(f"Con cache (1.ª vez)  : {medir(lambda: df.count()):.3f} s  (materializa la cache)")
print(f"Con cache (2.ª vez)  : {medir(lambda: df.count()):.3f} s")

**Interpretación.** La segunda ejecución con cache debería ser más rápida. Mientras la cache existe, puede observarse en la pestaña *Storage* de la Spark UI. Con datasets pequeños la diferencia es reducida; con datasets grandes es considerable.

## 7. WordCount

El conteo de palabras es el ejemplo clásico de procesamiento distribuido derivado de MapReduce. Si el dataset activo tiene una columna de texto libre (rol `texto`), se analiza esa columna; en caso contrario se usa el archivo `data/ejemplo/texto_bigdata.txt`.

In [ ]:
import re
STOPWORDS = {"de","la","el","en","y","a","los","las","que","un","una","por","con","para","se",
             "es","del","al","o","como","su","sus","lo","the","and","of","to"}

if TXT:
    print("Columna de texto analizada:", TXT[0])
    lineas = df.select(TXT[0]).dropna().rdd.map(lambda fila: str(fila[0]))
else:
    print("Archivo de texto analizado: data/ejemplo/texto_bigdata.txt")
    lineas = sc.textFile(str(ruta_absoluta("data/ejemplo/texto_bigdata.txt")))

palabras = (lineas
            .flatMap(lambda l: re.findall(r"\w+", l.lower()))     # línea -> palabras
            .filter(lambda p: p not in STOPWORDS and len(p) > 2)
            .map(lambda p: (p, 1))
            .reduceByKey(lambda a, b: a + b))

for palabra, n in palabras.takeOrdered(15, key=lambda par: -par[1]):
    print(f"{palabra:<20}{n:>6}")

## Ejercicios propuestos

1. Con la API de RDD, contar cuántos valores distintos tiene la columna `COLUMNA` (sugerencia: `distinct()` y `count()`).
2. Calcular con RDD la suma de la primera columna numérica (`NUM[0]`), ignorando los nulos. Comparar con `df.agg(F.sum(...))`.
3. Ejecutar `df.repartition(8)` seguido de un `count()` y describir en la Spark UI cuántas tareas se crearon.
4. Explicar, con base en el plan de `explain()`, por qué `orderBy` es una transformación amplia.

In [ ]:
df.unpersist()
spark.stop()

## Referencias

Armbrust, M., Xin, R. S., Lian, C., Huai, Y., Liu, D., Bradley, J. K., Meng, X., Kaftan, T., Franklin, M. J., Ghodsi, A., & Zaharia, M. (2015). Spark SQL: Relational data processing in Spark. En *Proceedings of the 2015 ACM SIGMOD International Conference on Management of Data* (pp. 1383–1394). ACM. https://doi.org/10.1145/2723372.2742797

Chambers, B., & Zaharia, M. (2018). *Spark: The definitive guide. Big data processing made simple*. O'Reilly Media.

The Apache Software Foundation. (2026a). *Spark overview* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/

The Apache Software Foundation. (2026b). *Spark SQL, DataFrames and Datasets guide* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/sql-programming-guide.html

Zaharia, M., Chowdhury, M., Das, T., Dave, A., Ma, J., McCauley, M., Franklin, M. J., Shenker, S., & Stoica, I. (2012). Resilient distributed datasets: A fault-tolerant abstraction for in-memory cluster computing. En *Proceedings of the 9th USENIX Symposium on Networked Systems Design and Implementation* (pp. 15–28). USENIX Association.